# Part 3: Signal Isolation & Quality Control

This notebook combines signal isolation (autofluorescence removal) with integrated quality control.

## Workflow Options

| Approach | Description | Best For |
|----------|-------------|----------|
| **Claude-Guided** | AI assistant suggests parameters, learns from success | New users, complex datasets |
| **Interactive Tuners** | Widget-based parameter adjustment | Quick iteration, visual feedback |
| **Python API** | Direct programmatic control | Automation, scripting |

## New Features (v1.1.0)

- **Parameter Learning**: System learns from successful parameters, improving recommendations over time
- **Integrated QC**: Quality assessment at each processing step
- **Advanced Denoising**: N2V, NLM, BM3D-lite, and adaptive methods
- **Claude Code Integration**: AI-assisted parameter selection via MCP server

## 1. Setup and Configuration

*Run this section every time the notebook is started.*

In [ ]:
# Auto-reload modules during development
%load_ext autoreload
%autoreload 2
%matplotlib widget

# Standard imports
import os
import json
import warnings
from pathlib import Path
from datetime import datetime
from glob import glob

import numpy as np
import dask.array as da
import matplotlib.pyplot as plt
import tifffile as tiff
import stackview
from skimage.io import imread_collection
from skimage.io.collection import alphanumeric_key
from natsort import natsorted

# KINTSUGI modules
from kintsugi.qc import ImageQC, CellQC, MarkerQC
from kintsugi.denoise import (
    adaptive_denoise,
    denoise_median,
    denoise_nlm,
    denoise_bilateral,
    estimate_noise_level,
)

# Legacy visualization (still useful)
import Kview
import Kutils

warnings.filterwarnings('ignore')

# =============================================================================
# GPU STATUS CHECK
# =============================================================================
from kintsugi.gpu import get_gpu_manager

gpu = get_gpu_manager()
print(gpu.summary())
print("\nImports successful")

### 1.1 Claude Code Integration (Optional)

If you created this project with `kintsugi init`, Claude Code is already configured (see Notebook 1 setup).

**Available Claude Tools:**
- `load_channel` - Load channel for analysis
- `suggest_parameters` - Get AI parameter recommendations
- `subtract_blank` - Apply autofluorescence subtraction
- `denoise_advanced` - Apply advanced denoising
- `assess_quality` - Run quality assessment
- `approve_and_learn` - Record successful parameters

In [ ]:
# =============================================================================
# PROJECT SETUP
# =============================================================================
from pathlib import Path
from kintsugi.project import init_project

# Choose your project: uncomment ONE of the following

# Option 1: Test Project (mini 2x2 dataset - fast iteration/testing)
TEST_PROJECT_DIR = Path("/blue/maigan/smith6jt/KINTSUGI/test_data/mini_project")

# Option 2: Full Project (real experimental data - production processing)  
FULL_PROJECT_DIR = Path("/blue/maigan/smith6jt/KINTSUGI_Projects/CODEX_SP_LN/1904CC1-1L")

# Select which project to use (change this to switch projects)
USE_TEST_PROJECT = False  # Set to True for test project, False for full project

# Select the active project directory
PROJECT_DIR = TEST_PROJECT_DIR if USE_TEST_PROJECT else FULL_PROJECT_DIR
PROJECT_NAME = "Mini Project" if USE_TEST_PROJECT else "1904CC1-1L"

print(f"Selected project: {PROJECT_NAME}")
print(f"Directory: {PROJECT_DIR}")

# Initialize project - will prompt if directory is non-empty
project = init_project(PROJECT_DIR, name=PROJECT_NAME, description="Signal isolation and QC")

# Check if project creation was cancelled
if project is None:
    raise RuntimeError("Project creation was cancelled. Please re-run this cell after deciding.")

# Use canonical project paths
image_dir = project.raw_dir
stitch_dir = project.paths.stitched
edf_dir = project.paths.edf
reg_dir = project.paths.registered
sig_dir = project.paths.signal_isolated
proc_dir = sig_dir  # Processed outputs go directly to signal_isolated
params_dir = project.paths.configs / "processing_parameters"
data_dir = project.root

for d in [reg_dir, sig_dir, params_dir]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"EDF images: {edf_dir}")
print(f"Registered images: {reg_dir}")
print(f"Signal Isolation output: {sig_dir}")
print(f"Parameters: {params_dir}")

# Scan for available channels in registered directory
def scan_registered_channels(reg_dir):
    """Scan registered directory for available channels organized by cycle."""
    channels_by_cycle = {}
    all_channels = {}
    
    reg_path = Path(reg_dir)
    for cycle_dir in sorted(reg_path.glob("cyc*")):
        if cycle_dir.is_dir():
            cycle_name = cycle_dir.name
            channels_by_cycle[cycle_name] = []
            
            for tif_file in sorted(cycle_dir.glob("*.tif")):
                channel_name = tif_file.stem
                channels_by_cycle[cycle_name].append(channel_name)
                # Store path for each unique channel
                if channel_name not in all_channels:
                    all_channels[channel_name] = tif_file
    
    return channels_by_cycle, all_channels

# Scan registered images
channels_by_cycle, channel_paths = scan_registered_channels(reg_dir)

print(f"\nFound {len(channel_paths)} unique channels across {len(channels_by_cycle)} cycles:")
for cycle, channels in list(channels_by_cycle.items())[:5]:
    print(f"  {cycle}: {channels}")
if len(channels_by_cycle) > 5:
    print(f"  ... and {len(channels_by_cycle) - 5} more cycles")


## 2. Load and Explore Channels

Extract single-channel TIFFs from registered OME-TIFFs and explore the data.

### 2.1 Extract Single-Channel TIFFs

*Run once to extract channels from registered OME-TIFFs.*

In [ ]:
import shutil
from pathlib import Path

# Configuration
start_cycle = 1
end_cycle = 9  # Adjust to your number of cycles
drop_duplicates = True  # Skip duplicate DAPI channels (keep only first)
drop_empty = True  # Skip empty/blank channels
dup_name = 'DAPI'
empty_names = ['Empty', 'Blank']  # Patterns to skip

def extract_channels_from_registered(reg_dir, sig_dir, start_cycle, end_cycle,
                                      drop_duplicates=True, drop_empty=True,
                                      dup_name='DAPI', empty_names=['Empty', 'Blank']):
    """
    Extract/copy channels from registered cycle directories to signal_isolated.
    
    Handles the structure: registered/cycXX/channel_name.tif
    Skips duplicate DAPI channels and empty/blank channels.
    """
    extracted = []
    skipped = []
    seen_dapi = False
    
    reg_path = Path(reg_dir)
    sig_path = Path(sig_dir)
    sig_path.mkdir(parents=True, exist_ok=True)
    
    for cycle in range(start_cycle, end_cycle + 1):
        cycle_dir = reg_path / f"cyc{cycle:02d}"
        
        if not cycle_dir.exists():
            print(f"Warning: {cycle_dir} not found, skipping")
            continue
        
        for tif_file in sorted(cycle_dir.glob("*.tif")):
            channel_name = tif_file.stem
            
            # Skip empty/blank channels
            if drop_empty:
                if any(empty.lower() in channel_name.lower() for empty in empty_names):
                    skipped.append((cycle, channel_name, "empty/blank"))
                    continue
            
            # Skip duplicate DAPI (keep only first occurrence)
            if drop_duplicates and dup_name.lower() in channel_name.lower():
                if seen_dapi:
                    skipped.append((cycle, channel_name, "duplicate DAPI"))
                    continue
                seen_dapi = True
            
            # Copy to signal_isolated directory
            output_path = sig_path / f"{channel_name}.tif"
            
            # Skip if already exists with same size
            if output_path.exists():
                if output_path.stat().st_size == tif_file.stat().st_size:
                    extracted.append(channel_name)
                    continue
            
            shutil.copy2(tif_file, output_path)
            extracted.append(channel_name)
            print(f"  Cycle {cycle}: {channel_name}")
    
    print(f"\nExtracted {len(extracted)} channels")
    if skipped:
        print(f"Skipped {len(skipped)} channels:")
        for cyc, name, reason in skipped[:10]:
            print(f"  cyc{cyc:02d}/{name}: {reason}")
        if len(skipped) > 10:
            print(f"  ... and {len(skipped) - 10} more")
    
    return extracted

# Run extraction
print("Extracting channels from registered directory...")
print(f"Source: {reg_dir}")
print(f"Destination: {sig_dir}")
print(f"Cycles: {start_cycle}-{end_cycle}")
print(f"Skip duplicates: {drop_duplicates}, Skip empty: {drop_empty}")
print()

extracted_channels = extract_channels_from_registered(
    reg_dir, sig_dir, start_cycle, end_cycle,
    drop_duplicates, drop_empty, dup_name, empty_names
)


### 2.2 Load Channels for Processing

In [ ]:
# Load channels from signal_isolated directory
# Handles images of different sizes by loading into dictionary (not stacked array)
from pathlib import Path
from natsort import natsorted
from glob import glob
from skimage.io import imread
import dask.array as da
import numpy as np

# Fallback if sig_dir not defined
if 'sig_dir' not in dir():
    possible_paths = [
        Path("/blue/maigan/smith6jt/KINTSUGI_Projects/CODEX_SP_LN/1904CC1-1L/data/processed/signal_isolated"),
        Path("data/processed/signal_isolated"),
    ]
    for p in possible_paths:
        if p.exists():
            sig_dir = p
            print(f"Using signal_isolated directory: {sig_dir}")
            break
    else:
        raise RuntimeError("Could not find signal_isolated directory. Run extraction cell first.")

# Find all channel files
file_list = natsorted(glob(str(Path(sig_dir) / '*.tif')))
marker_names = [Path(f).stem for f in file_list]

if file_list:
    print(f"Found {len(file_list)} channels in {sig_dir}")
    
    # Check if images have same dimensions
    sample_shapes = []
    for f in file_list[:5]:
        img = imread(f)
        sample_shapes.append(img.shape)
    
    shapes_match = len(set(sample_shapes)) == 1
    
    if shapes_match:
        # All same size - can stack into array
        print(f"All images same size: {sample_shapes[0]}")
        chunk_size = 1024
        
        # Load all images
        images = [imread(f) for f in file_list]
        markers_array = np.stack(images, axis=0)
        markers = da.from_array(markers_array, chunks=(1, chunk_size, chunk_size))
        
        # Create dictionary for easy access
        marker_dict = {name: markers[i] for i, name in enumerate(marker_names)}
        
        print(f"Loaded as stacked array: {markers.shape}")
    else:
        # Different sizes - load individually into dictionary
        print(f"Images have different sizes - loading individually")
        print(f"  Sample shapes: {sample_shapes}")
        
        marker_dict = {}
        for name, fpath in zip(marker_names, file_list):
            img = imread(fpath)
            marker_dict[name] = da.from_array(img, chunks=(1024, 1024))
        
        # No stacked array available
        markers = None
        print(f"Loaded {len(marker_dict)} channels into marker_dict")
    
    print(f"\nChannels available:")
    for i, name in enumerate(marker_names):
        shape = marker_dict[name].shape
        print(f"  {i+1:2d}. {name}: {shape}")
        if i >= 14:
            print(f"  ... and {len(marker_names) - 15} more")
            break
else:
    print("No TIF files found in signal_isolated directory.")
    print(f"Run the extraction cell (Section 2.1) first.")
    marker_names = []
    marker_dict = {}


### 2.3 Explore Channels

In [ ]:
# Interactive channel browser
if 'marker_dict' in dir() and marker_dict:
    stackview.switch(marker_dict, zoom_factor=0.1, colormap='turbo')

In [ ]:
# Optional: Crop to region of interest
if 'markers' in dir():
    stackview.crop(markers, zoom_factor=0.06, colormap='turbo', continuous_update=False)

In [ ]:
# Apply crop values (adjust based on crop widget above)
crop_x1, crop_x2 = 0, None  # Left, Right
crop_y1, crop_y2 = 0, None  # Top, Bottom

if 'marker_dict' in dir() and marker_dict:
    marker_dict_cropped = {
        name: img[crop_y1:crop_y2, crop_x1:crop_x2]
        for name, img in marker_dict.items()
    }
    
    # Make channels available as variables
    globals().update(marker_dict_cropped)
    
    sample_shape = list(marker_dict_cropped.values())[0].shape
    print(f"Cropped dimensions: {sample_shape}")

## 3. Signal Isolation

Remove autofluorescence and noise to isolate true signal.

Choose your preferred approach:
- **3A**: Claude-Guided (AI recommendations)
- **3B**: Interactive Tuners (widget-based)
- **3C**: Direct Python API

### 3A. Claude-Guided Workflow

With Claude Code configured, ask Claude to help with signal isolation:

```
User: "Load the CD3e channel and suggest blank subtraction parameters"
Claude: [Uses load_channel and suggest_with_learning tools]
Claude: "Based on analysis, I recommend using Blank1b with scale_factor=1.2..."

User: "Apply those parameters and show me the result"
Claude: [Uses subtract_blank tool]

User: "That looks good, approve and save the parameters"
Claude: [Uses approve_and_learn to record for future use]
```

The system learns from successful parameters, so recommendations improve over time.

### 3B. Interactive Tuner Workflow

#### 3B.1 Blank Subtraction

Use the interactive widget to find optimal blank subtraction parameters.

In [ ]:
# Interactive blank subtraction tuner
# - Choose signal channel (im1) and blank channel (im2)
# - Adjust blank_clip_factor and blank_scale_factor
# - Enable smooth_low/smooth_high for filtering

Kview2.interact(
    Kutils.ini_params, 
    context=globals(), 
    zoom_factor=0.05, 
    colormap='turbo',
    min_value=0, max_value=20000, step=200,
    min_value_float=0, max_value_float=3.0, step_float=0.1,
    smooth_low=False, low_size=2,
    smooth_high=False, high_size=2,
    erosion=1,
    low_percentile=50, high_percentile=50,
    continuous_update=False,
    display_min=0, display_max=65535
)

In [ ]:
# Apply blank subtraction with determined parameters
signal_channel = 'CD3e'  # Change to your channel
blank_channel = 'Blank1b'  # Change to your blank

# Parameters from interactive tuning
blank_params = {
    'blank_clip_factor': 20000,
    'blank_scale_factor': 1.4,
    'smooth_low': False,
    'low_size': 2,
    'low_percentile': 80,
    'smooth_high': False,
    'high_size': 2,
    'high_percentile': 90,
    'erosion': 1,
}

# Apply subtraction
signal_data = marker_dict_cropped[signal_channel]
blank_data = marker_dict_cropped[blank_channel]

signal_sub1 = Kutils.ini_params(
    signal_data, blank_data,
    blank_params['blank_clip_factor'],
    blank_params['blank_scale_factor'],
    blank_params['smooth_low'],
    blank_params['low_size'],
    blank_params['low_percentile'],
    blank_params['smooth_high'],
    blank_params['high_size'],
    blank_params['high_percentile'],
    blank_params['erosion'],
    View_original=False
)

plt.figure()
stackview.imshow(signal_sub1, colormap='turbo', colorbar=True)

#### 3B.2 Denoising

Apply denoising using new advanced methods or legacy filters.

In [ ]:
# Analyze noise level to determine best approach
if 'signal_sub1' in dir():
    # Compute to numpy if dask array
    img_np = signal_sub1.compute() if hasattr(signal_sub1, 'compute') else signal_sub1
    
    noise_level = estimate_noise_level(img_np, method='mad')
    print(f"Estimated noise level (MAD): {noise_level:.2f}")
    
    # Suggest denoising approach
    if noise_level > 100:
        print("Recommendation: Strong denoising needed (NLM or N2V)")
    elif noise_level > 50:
        print("Recommendation: Moderate denoising (NLM or bilateral)")
    else:
        print("Recommendation: Light denoising (median or bilateral)")

In [ ]:
# Option A: Adaptive denoising (automatically selects best method)
denoise_used = True

if denoise_used and 'signal_sub1' in dir():
    img_np = signal_sub1.compute() if hasattr(signal_sub1, 'compute') else signal_sub1
    
    # Adaptive denoising - auto-selects method and parameters
    denoised, method_used = adaptive_denoise(
        img_np, 
        strength='auto',  # 'light', 'moderate', 'strong', or 'auto'
        return_method=True
    )
    
    print(f"Method used: {method_used}")
    
    plt.figure()
    stackview.imshow(denoised, colormap='turbo', colorbar=True)

In [ ]:
# Option B: Manual denoising selection
# Uncomment the method you want to use

if 'signal_sub1' in dir():
    img_np = signal_sub1.compute() if hasattr(signal_sub1, 'compute') else signal_sub1
    
    # Median filter (good for salt-and-pepper noise)
    # denoised = denoise_median(img_np, size=3)
    
    # Non-local means (good for preserving edges)
    # denoised = denoise_nlm(img_np, patch_size=7, patch_distance=11, h=None)
    
    # Bilateral (edge-preserving smoothing)
    # denoised = denoise_bilateral(img_np, sigma_color=None, sigma_spatial=5)
    
    pass

In [ ]:
# Option C: Legacy interactive denoising
# Uses percentile, uniform, and median filters

Kview2.interact(
    Kutils.denoise, 
    context=globals(), 
    zoom_factor=0.05, 
    colormap='turbo',
    min_value=1, max_value=50, step=1,
    min_value_float=0, max_value_float=10.0, step_float=0.1,
    continuous_update=False,
    display_min=0, display_max=65535
)

#### 3B.3 Contrast Enhancement (CLAHE)

In [ ]:
# Interactive CLAHE tuning
Kview2.interact(
    Kutils.CLAHE, 
    context=globals(), 
    zoom_factor=0.03, 
    colormap='turbo',
    continuous_update=False,
    min_value=10, max_value=300, step=1,
    min_value_float=0.01, max_value_float=0.2, step_float=0.01,
    display_min=0, display_max=65535
)

In [ ]:
# Apply CLAHE with determined parameters
clahe_used = False

if clahe_used:
    clahe_params = {
        'clip_limit': 0.02,
        'tileGridSize': 300,
        'nbins': 128,
    }
    
    # Use denoised image or signal_sub1
    input_image = denoised if 'denoised' in dir() else signal_sub1
    img_np = input_image.compute() if hasattr(input_image, 'compute') else input_image
    
    signal_clahe = Kutils.CLAHE(
        img_np,
        clip_limit=clahe_params['clip_limit'],
        tileGridSize=clahe_params['tileGridSize'],
        nbins=clahe_params['nbins'],
        View_original=False
    )
    
    plt.figure()
    stackview.imshow(signal_clahe, colormap='turbo', colorbar=True)

#### 3B.4 Background Cleaning

In [ ]:
# Interactive background cleaning
Kview2.interact(
    Kutils.clean, 
    context=globals(), 
    zoom_factor=0.04, 
    colormap='turbo',
    smooth=False, 
    remove_small=False, 
    footprint=3, 
    small_size=50,
    continuous_update=False,
    min_value=0, max_value=30000, step=100,
    min_value_float=0, max_value_float=10.0, step_float=0.1,
    display_min=0, display_max=65535
)

In [ ]:
# Apply cleaning with determined parameters
clean_used = True

if clean_used:
    clean_params = {
        'backgrnd_thresh': 1000,
        'smooth': True,
        'smooth_thresh': 1000,
        'footprint': 3,
        'remove_small': True,
        'small_size': 50,
    }
    
    # Use best available processed image
    if 'signal_clahe' in dir():
        input_image = signal_clahe
    elif 'denoised' in dir():
        input_image = denoised
    else:
        input_image = signal_sub1
    
    img_np = input_image.compute() if hasattr(input_image, 'compute') else input_image
    
    signal_final = Kutils.clean(
        img_np,
        backgrnd_thresh=clean_params['backgrnd_thresh'],
        smooth_thresh=clean_params['smooth_thresh'],
        smooth=clean_params['smooth'],
        remove_small=clean_params['remove_small'],
        small_size=clean_params['small_size'],
        footprint=clean_params['footprint'],
        View_original=False
    )
    
    plt.figure()
    stackview.imshow(signal_final, colormap='turbo', colorbar=True)

## 4. Quality Assessment

Assess the quality of processed images using the integrated QC module.

In [ ]:
# Run quality assessment on final processed image
if 'signal_final' in dir():
    qc = ImageQC()
    
    # Assess quality
    result = qc.assess(
        signal_final,
        marker=signal_channel if 'signal_channel' in dir() else None,
        tissue='tonsil',  # Adjust to your tissue type
    )
    
    print("=" * 60)
    print("QUALITY ASSESSMENT RESULTS")
    print("=" * 60)
    print(f"Overall Quality Score: {result.quality_score:.2f}")
    print(f"Passed QC: {result.passed}")
    print(f"\nMetrics:")
    for metric, value in result.metrics.items():
        print(f"  {metric}: {value:.4f}")
    
    if result.issues:
        print(f"\nIssues Detected:")
        for issue in result.issues:
            print(f"  - {issue}")
    
    if result.recommendations:
        print(f"\nRecommendations:")
        for rec in result.recommendations:
            print(f"  - {rec}")

In [ ]:
# Compare original vs processed with quality metrics
if 'signal_final' in dir() and 'signal_channel' in dir():
    original = marker_dict_cropped[signal_channel]
    original_np = original.compute() if hasattr(original, 'compute') else original
    
    # Assess both
    qc = ImageQC()
    orig_result = qc.assess(original_np)
    proc_result = qc.assess(signal_final)
    
    print("Quality Comparison:")
    print(f"  Original SNR:  {orig_result.metrics.get('snr', 0):.2f}")
    print(f"  Processed SNR: {proc_result.metrics.get('snr', 0):.2f}")
    print(f"  Improvement:   {(proc_result.metrics.get('snr', 0) / max(orig_result.metrics.get('snr', 1), 1) - 1) * 100:.1f}%")

## 5. Visualization and Comparison

In [ ]:
# Curtain comparison view
if 'signal_final' in dir() and 'signal_channel' in dir():
    original = marker_dict_cropped[signal_channel]
    original_np = original.compute() if hasattr(original, 'compute') else original
    
    # Crop for faster display
    crop_factor = 5
    h, w = signal_final.shape
    y1, y2 = 0, h // crop_factor
    x1, x2 = 0, w // crop_factor
    
    stackview.curtain(
        signal_final[y1:y2, x1:x2],
        original_np[y1:y2, x1:x2],
        alpha=1.0,
        zoom_factor=0.4,
        colormap='turbo',
        curtain_colormap='turbo'
    )

In [ ]:
# Toggle view with DAPI overlay
if 'signal_final' in dir() and 'DAPI' in marker_dict_cropped:
    original = marker_dict_cropped[signal_channel]
    dapi = marker_dict_cropped['DAPI']
    
    original_np = original.compute() if hasattr(original, 'compute') else original
    dapi_np = dapi.compute() if hasattr(dapi, 'compute') else dapi
    
    # Crop
    crop_factor = 5
    h, w = signal_final.shape
    y1, y2 = 0, h // crop_factor
    x1, x2 = 0, w // crop_factor
    
    Kview2.switch(
        {
            "Original": original_np[y1:y2, x1:x2],
            "Processed": signal_final[y1:y2, x1:x2],
            "DAPI": dapi_np[y1:y2, x1:x2],
        },
        colormap=["pure_magenta", "pure_green", "pure_blue"],
        zoom_factor=0.4,
        toggleable=True,
    )

## 6. Save Results and Parameters

In [ ]:
# Collect all parameters used
processing_params = {
    'timestamp': datetime.now().isoformat(),
    'signal_channel': signal_channel if 'signal_channel' in dir() else None,
    'blank_channel': blank_channel if 'blank_channel' in dir() else None,
    'crop': {
        'x1': crop_x1, 'x2': crop_x2,
        'y1': crop_y1, 'y2': crop_y2,
    },
    'blank_subtraction': blank_params if 'blank_params' in dir() else None,
    'denoising': {
        'used': denoise_used if 'denoise_used' in dir() else False,
        'method': method_used if 'method_used' in dir() else None,
    },
    'clahe': {
        'used': clahe_used if 'clahe_used' in dir() else False,
        'params': clahe_params if 'clahe_params' in dir() else None,
    },
    'cleaning': {
        'used': clean_used if 'clean_used' in dir() else False,
        'params': clean_params if 'clean_params' in dir() else None,
    },
    'quality': {
        'score': result.quality_score if 'result' in dir() else None,
        'passed': result.passed if 'result' in dir() else None,
    },
}

# Save parameters
if 'signal_channel' in dir():
    params_file = params_dir / f"{signal_channel}_params.json"
    with open(params_file, 'w') as f:
        json.dump(processing_params, f, indent=2, default=str)
    print(f"Parameters saved to: {params_file}")

In [ ]:
# Save processed image
from skimage.exposure import rescale_intensity, match_histograms

if 'signal_final' in dir() and 'signal_channel' in dir():
    # Normalize and scale
    def generate_reference_histogram(shape):
        """Generate reference histogram for consistent output."""
        uniform = np.random.rand(*shape)
        exponential = -np.log(1 - uniform)
        normalized = np.clip(exponential, 0, 100) / 100
        return (normalized * 65535).astype(np.uint16)
    
    ref_hist = generate_reference_histogram(signal_final.shape)
    matched = match_histograms(signal_final.astype(np.float32), ref_hist)
    saved_image = rescale_intensity(matched, out_range=(0, 1000)).astype(np.uint16)
    
    # Save
    output_path = proc_dir / f"{signal_channel}.tif"
    tiff.imwrite(str(output_path), saved_image)
    print(f"Image saved to: {output_path}")
    
    # Preview
    stackview.insight(saved_image)

## 7. Parameter Learning (Optional)

Record successful parameters for future use with similar tissue/marker combinations.

In [ ]:
# Record successful parameters to learning database
try:
    from kintsugi.mcp.tools.learning import ParameterLearningEngine
    
    # Initialize learning engine
    engine = ParameterLearningEngine(str(data_dir))
    
    # Record if QC passed
    if 'result' in dir() and result.passed:
        tissue_type = 'tonsil'  # Adjust to your tissue
        
        # Record blank subtraction parameters
        if 'blank_params' in dir():
            engine.record_parameters(
                operation='blank_subtraction',
                tissue_type=tissue_type,
                marker_name=signal_channel,
                parameters=blank_params,
                quality_score=result.quality_score,
            )
        
        # Record cleaning parameters
        if 'clean_params' in dir() and clean_used:
            engine.record_parameters(
                operation='background_cleaning',
                tissue_type=tissue_type,
                marker_name=signal_channel,
                parameters=clean_params,
                quality_score=result.quality_score,
            )
        
        print(f"Parameters recorded for {signal_channel} ({tissue_type})")
        print("Future recommendations will use this data.")
    else:
        print("QC did not pass - parameters not recorded.")
        
except ImportError:
    print("Parameter learning requires: pip install kintsugi[claude]")

In [ ]:
# Get recommendations for a new channel
try:
    from kintsugi.mcp.tools.learning import ParameterLearningEngine
    
    engine = ParameterLearningEngine(str(data_dir))
    
    # Get recommendations
    new_marker = 'CD20'  # Change to your marker
    tissue_type = 'tonsil'
    
    rec = engine.recommend_parameters(
        operation='blank_subtraction',
        tissue_type=tissue_type,
        marker_name=new_marker,
    )
    
    if rec['found']:
        print(f"Recommendations for {new_marker} ({tissue_type}):")
        print(f"  Confidence: {rec['confidence']:.2f}")
        print(f"  Based on: {rec['sample_count']} successful runs")
        print(f"  Parameters: {rec['recommended_parameters']}")
    else:
        print(f"No learned parameters for {new_marker} ({tissue_type})")
        print("Use interactive tuning to establish baseline.")
        
except ImportError:
    print("Parameter learning requires: pip install kintsugi[claude]")

## 8. Batch Processing

Apply determined parameters to multiple channels.

In [ ]:
# Define channels to process with same parameters
channels_to_process = [
    # ('signal_channel', 'blank_channel'),
    # ('CD3e', 'Blank1b'),
    # ('CD20', 'Blank1b'),
    # ('CD8', 'Blank1b'),
]

# Reuse parameters from tuning
batch_blank_params = blank_params if 'blank_params' in dir() else None
batch_clean_params = clean_params if 'clean_params' in dir() else None

if channels_to_process and batch_blank_params:
    from tqdm import tqdm
    
    results = []
    qc = ImageQC()
    
    for sig_ch, blank_ch in tqdm(channels_to_process, desc="Processing channels"):
        try:
            # Load
            sig_data = marker_dict_cropped[sig_ch]
            blank_data = marker_dict_cropped[blank_ch]
            
            sig_np = sig_data.compute() if hasattr(sig_data, 'compute') else sig_data
            blank_np = blank_data.compute() if hasattr(blank_data, 'compute') else blank_data
            
            # Process
            processed = Kutils.ini_params(
                sig_np, blank_np,
                batch_blank_params['blank_clip_factor'],
                batch_blank_params['blank_scale_factor'],
                batch_blank_params['smooth_low'],
                batch_blank_params['low_size'],
                batch_blank_params['low_percentile'],
                batch_blank_params['smooth_high'],
                batch_blank_params['high_size'],
                batch_blank_params['high_percentile'],
                batch_blank_params['erosion'],
                View_original=False
            )
            
            # Clean
            if batch_clean_params:
                processed = Kutils.clean(
                    processed,
                    backgrnd_thresh=batch_clean_params['backgrnd_thresh'],
                    smooth_thresh=batch_clean_params['smooth_thresh'],
                    smooth=batch_clean_params['smooth'],
                    remove_small=batch_clean_params['remove_small'],
                    small_size=batch_clean_params['small_size'],
                    footprint=batch_clean_params['footprint'],
                    View_original=False
                )
            
            # QC
            qc_result = qc.assess(processed)
            
            # Save
            output_path = proc_dir / f"{sig_ch}.tif"
            tiff.imwrite(str(output_path), processed.astype(np.uint16))
            
            results.append({
                'channel': sig_ch,
                'passed': qc_result.passed,
                'score': qc_result.quality_score,
            })
            
        except Exception as e:
            results.append({
                'channel': sig_ch,
                'passed': False,
                'error': str(e),
            })
    
    # Summary
    print("\nBatch Processing Summary:")
    passed = sum(1 for r in results if r.get('passed', False))
    print(f"  Passed QC: {passed}/{len(results)}")
    
    for r in results:
        status = 'PASS' if r.get('passed') else 'FAIL'
        score = r.get('score', 'N/A')
        print(f"  {r['channel']}: {status} (score: {score})")

## 9. Merge and Export

Merge all processed channels into final OME-TIFF.

In [ ]:
# Merge all processed channels
sample_id = 'SAMPLE_PROJECT'  # Change to your sample ID
pixel_size = 0.3774  # um/pixel

proc_files = natsorted(glob(str(proc_dir / '*.tif')))
proc_files = [f for f in proc_files if not f.endswith('.ome.tif')]  # Exclude existing OME-TIFFs

if proc_files:
    proc_col = imread_collection(proc_files, conserve_memory=True)
    proc_stack = np.asarray(proc_col)
    proc_names = [Path(f).stem for f in proc_files]
    
    print(f"Merging {len(proc_names)} channels:")
    print(f"  {proc_names}")
    print(f"  Shape: {proc_stack.shape}")
    
    # Create OME-TIFF metadata
    metadata = {
        'axes': 'CYX',
        'PhysicalSizeX': pixel_size,
        'PhysicalSizeXUnit': 'um',
        'PhysicalSizeY': pixel_size,
        'PhysicalSizeYUnit': 'um',
        'Channel': [{'Name': name} for name in proc_names],
    }
    
    # Save
    output_path = proc_dir / f"{sample_id}.ome.tif"
    tiff.imwrite(
        str(output_path),
        proc_stack,
        metadata=metadata,
        photometric='minisblack',
        compression='zlib',
    )
    
    print(f"\nSaved to: {output_path}")
else:
    print("No processed files found.")

## Next Steps

1. **Notebook 4**: Segmentation & Spatial Analysis
   - Use the merged OME-TIFF for cell segmentation
   - Extract features and perform clustering

2. **Parameter Refinement**:
   - Use Claude Code to refine parameters for problematic channels
   - Build up learning database for faster future processing

3. **Quality Reports**:
   - Review parameters in `Processing_parameters/` directory
   - Check QC scores for all channels